# 🚀 3D Gaussian Splatting - Complete Training

## NeRF Synthetic Dataset (Lego, Chair, Drums, etc.)

### All Paper Components:
- ✅ Camera Projection & Jacobians
- ✅ Gaussian Model (position, scale, rotation, opacity)
- ✅ Spherical Harmonics (view-dependent colors)
- ✅ 3D Covariance → 2D Projection
- ✅ Densification & Pruning
- ✅ L1 + SSIM Loss with PSNR tracking
- ✅ Checkpointing & Resume

---

In [ ]:
# ============================================
# SETUP
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os, zipfile
from torch.optim import Adam
from tqdm import tqdm
import imageio.v2 as imageio

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# 🎯 CHOOSE YOUR SCENE HERE:
SCENE = 'Lego'  # Options: Lego, Chair, Drums, Ficus, Hotdog, Materials, Mic, Ship

config = {
    'num_iterations': 10000,
    'checkpoint_every': 2000,
    'display_every': 500,
    'densify_from': 500,
    'densify_until': 7000,
    'densify_every': 200,
    'densify_grad_thresh': 0.0002,
    'prune_opacity_thresh': 0.005,
    'lr_position': 0.00016,
    'lr_scale': 0.005,
    'lr_rotation': 0.001,
    'lr_opacity': 0.05,
    'lr_sh': 0.0025,
    'lambda_ssim': 0.2,
    'sh_degree': 2,
    'initial_gaussians': 50000,
    'max_images': 25,
    'downscale': 4,
    'base_dir': f'/content/drive/MyDrive/3DGS/{SCENE}',
}
config['checkpoint_dir'] = f"{config['base_dir']}/checkpoints"
config['output_dir'] = f"{config['base_dir']}/outputs"

for d in [config['base_dir'], config['checkpoint_dir'], config['output_dir']]:
    os.makedirs(d, exist_ok=True)
print(f"✅ Scene: {SCENE}")
print(f"✅ Output: {config['base_dir']}")

In [ ]:
# ============================================
# DOWNLOAD DATASET
# ============================================
data_dir = '/content/data'
os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(f'{data_dir}/Synthetic_NeRF'):
    print("📥 Downloading NeRF Synthetic dataset...")
    !wget -q --show-progress -O /content/nerf.zip \
        "https://dl.fbaipublicfiles.com/nsvf/dataset/Synthetic_NeRF.zip"
    print("📦 Extracting...")
    with zipfile.ZipFile('/content/nerf.zip', 'r') as z:
        z.extractall(data_dir)
    !rm /content/nerf.zip

DATASET_PATH = f'{data_dir}/Synthetic_NeRF/{SCENE}'
print(f"✅ Dataset ready: {DATASET_PATH}")

# Show available scenes
scenes = os.listdir(f'{data_dir}/Synthetic_NeRF')
print(f"📁 Available scenes: {scenes}")

In [ ]:
# ============================================
# CAMERA CLASS (Notebook 02)
# ============================================
class Camera:
    def __init__(self, R, T, fx, fy, cx, cy, width, height):
        self.R = R.float().to(device)
        self.T = T.float().to(device)
        self.fx, self.fy = float(fx), float(fy)
        self.cx, self.cy = float(cx), float(cy)
        self.width, self.height = int(width), int(height)
    
    def project(self, xyz):
        """Project 3D points to 2D."""
        t = -self.R @ self.T
        xyz_cam = (self.R @ xyz.T).T + t
        z = torch.clamp(xyz_cam[:, 2], min=0.01)
        u = self.fx * xyz_cam[:, 0] / z + self.cx
        v = self.fy * xyz_cam[:, 1] / z + self.cy
        return u, v, xyz_cam
    
    def get_view_dir(self, xyz):
        """Get view direction from camera to points."""
        return F.normalize(xyz - self.T.unsqueeze(0), dim=-1)
    
    def get_projection_jacobian(self, xyz_cam):
        """Jacobian for covariance projection."""
        z = torch.clamp(xyz_cam[:, 2:3], min=0.01)
        J = torch.zeros(xyz_cam.shape[0], 2, 3, device=device)
        J[:, 0, 0] = self.fx / z.squeeze()
        J[:, 0, 2] = -self.fx * xyz_cam[:, 0] / (z.squeeze() ** 2)
        J[:, 1, 1] = self.fy / z.squeeze()
        J[:, 1, 2] = -self.fy * xyz_cam[:, 1] / (z.squeeze() ** 2)
        return J

In [ ]:
# ============================================
# LOAD NSVF DATASET
# ============================================
def load_dataset(data_path, max_images, downscale):
    with open(f'{data_path}/intrinsics.txt', 'r') as f:
        intrinsics = [float(x) for x in f.read().split()]
    fx, fy, cx, cy = intrinsics[:4]
    
    rgb_dir = f'{data_path}/rgb'
    pose_dir = f'{data_path}/pose'
    img_files = sorted([f for f in os.listdir(rgb_dir) if f.endswith('.png')])[:max_images]
    
    images, cameras = [], []
    for img_file in tqdm(img_files, desc='Loading images'):
        img = imageio.imread(f'{rgb_dir}/{img_file}')
        if downscale > 1:
            img = img[::downscale, ::downscale]
        img = torch.from_numpy(img).float() / 255.0
        if img.shape[-1] == 4:
            alpha = img[..., 3:4]
            img = img[..., :3] * alpha + (1 - alpha)
        H, W = img.shape[:2]
        images.append(img.to(device))
        
        pose = np.loadtxt(f'{pose_dir}/{img_file.replace(".png", ".txt")}').reshape(4, 4)
        c2w = torch.tensor(pose, dtype=torch.float32)
        cameras.append(Camera(c2w[:3,:3].T, c2w[:3,3], 
                             fx/downscale, fy/downscale, cx/downscale, cy/downscale, W, H))
    
    print(f"✅ Loaded {len(images)} images ({H}x{W})")
    return images, cameras

images, cameras = load_dataset(DATASET_PATH, config['max_images'], config['downscale'])

# Preview
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    axes[i].imshow(images[i].cpu().numpy())
    axes[i].set_title(f'{SCENE} - View {i}')
    axes[i].axis('off')
plt.show()

In [ ]:
# ============================================
# SPHERICAL HARMONICS (Notebook 05)
# ============================================
C0 = 0.28209479177387814
C1 = 0.4886025119029199
C2 = [1.0925484305920792, -1.0925484305920792, 0.31539156525252, 
      -1.0925484305920792, 0.5462742152960396]

def eval_sh(degree, sh_coeffs, directions):
    """Evaluate spherical harmonics."""
    result = C0 * sh_coeffs[:, 0]
    if degree >= 1 and sh_coeffs.shape[1] > 1:
        x, y, z = directions[:, 0:1], directions[:, 1:2], directions[:, 2:3]
        result = result + C1 * (-y * sh_coeffs[:, 1] + z * sh_coeffs[:, 2] - x * sh_coeffs[:, 3])
        if degree >= 2 and sh_coeffs.shape[1] > 4:
            xx, yy, zz = x*x, y*y, z*z
            xy, xz, yz = x*y, x*z, y*z
            result = result + C2[0] * xy * sh_coeffs[:, 4]
            result = result + C2[1] * yz * sh_coeffs[:, 5]
            result = result + C2[2] * (3*zz - 1) * sh_coeffs[:, 6]
            result = result + C2[3] * xz * sh_coeffs[:, 7]
            result = result + C2[4] * (xx - yy) * sh_coeffs[:, 8]
    return result + 0.5

In [ ]:
# ============================================
# GAUSSIAN MODEL (Notebook 03)
# ============================================
class GaussianModel:
    def __init__(self, num_gaussians, sh_degree=2, device='cuda'):
        self.device = device
        self.sh_degree = sh_degree
        num_sh = (sh_degree + 1) ** 2
        
        # Learnable parameters
        self._xyz = nn.Parameter(torch.randn(num_gaussians, 3, device=device) * 2)
        self._log_scale = nn.Parameter(torch.ones(num_gaussians, 3, device=device) * -3)
        self._rotation = nn.Parameter(torch.zeros(num_gaussians, 4, device=device))
        self._rotation.data[:, 0] = 1.0
        self._opacity = nn.Parameter(torch.zeros(num_gaussians, 1, device=device))
        self._sh = nn.Parameter(torch.zeros(num_gaussians, num_sh, 3, device=device))
        self._sh.data[:, 0, :] = 0.5
        
        # For densification
        self.xyz_grad_accum = torch.zeros(num_gaussians, device=device)
        self.denom = torch.zeros(num_gaussians, device=device)
    
    @property
    def xyz(self): return self._xyz
    @property
    def scales(self): return torch.exp(self._log_scale)
    @property
    def rotations(self): return F.normalize(self._rotation, dim=-1)
    @property
    def opacity(self): return torch.sigmoid(self._opacity)
    
    def get_colors(self, view_dirs):
        return torch.clamp(eval_sh(self.sh_degree, self._sh, view_dirs), 0, 1)
    
    def get_covariance_3d(self):
        """Build 3D covariance from scale and rotation."""
        S = torch.diag_embed(self.scales)
        q = self.rotations
        r, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        R = torch.stack([
            torch.stack([1-2*(y*y+z*z), 2*(x*y-r*z), 2*(x*z+r*y)], dim=-1),
            torch.stack([2*(x*y+r*z), 1-2*(x*x+z*z), 2*(y*z-r*x)], dim=-1),
            torch.stack([2*(x*z-r*y), 2*(y*z+r*x), 1-2*(x*x+y*y)], dim=-1),
        ], dim=1)
        M = R @ S
        return M @ M.transpose(-1, -2)
    
    def get_covariance_2d(self, camera, xyz_cam):
        """Project 3D covariance to 2D screen space."""
        cov3d = self.get_covariance_3d()
        J = camera.get_projection_jacobian(xyz_cam)
        W = camera.R.unsqueeze(0)
        T = J @ W
        cov2d = T @ cov3d @ T.transpose(-1, -2)
        cov2d[:, 0, 0] += 0.3
        cov2d[:, 1, 1] += 0.3
        return cov2d
    
    def __len__(self): return self._xyz.shape[0]

In [ ]:
# ============================================
# DIFFERENTIABLE RENDERER (Notebook 04)
# ============================================
def render_gaussians(model, camera):
    """Render Gaussians to image."""
    width, height = camera.width, camera.height
    
    # Project
    u, v, xyz_cam = camera.project(model.xyz)
    
    # Get colors and opacity
    view_dirs = camera.get_view_dir(model.xyz)
    colors = model.get_colors(view_dirs)
    opacity = model.opacity.squeeze(-1)
    
    # Compute 2D covariance for radius
    cov2d = model.get_covariance_2d(camera, xyz_cam)
    radius = 3 * torch.sqrt(torch.max(cov2d[:, 0, 0], cov2d[:, 1, 1]))
    
    # Valid Gaussians in view
    valid = (xyz_cam[:, 2] > 0.01) & (u > -radius) & (u < width + radius) & (v > -radius) & (v < height + radius)
    
    # Filter
    u_v, v_v = u[valid], v[valid]
    colors_v = colors[valid]
    opacity_v = opacity[valid]
    
    # Pixel indices
    pixel_idx = (v_v.long().clamp(0, height-1) * width + u_v.long().clamp(0, width-1))
    
    # Accumulate
    weights = opacity_v
    weighted_colors = colors_v * weights.unsqueeze(-1)
    
    accumulated_color = torch.zeros(height * width, 3, device=device)
    accumulated_weight = torch.zeros(height * width, device=device)
    
    accumulated_color.scatter_add_(0, pixel_idx.unsqueeze(-1).expand(-1, 3), weighted_colors)
    accumulated_weight.scatter_add_(0, pixel_idx, weights)
    
    # Blend with background
    accumulated_weight = accumulated_weight.unsqueeze(-1).clamp(min=1e-6, max=1.0)
    image = torch.ones(height * width, 3, device=device)
    image = image * (1 - accumulated_weight) + accumulated_color / accumulated_weight.clamp(min=1e-6) * accumulated_weight
    
    return torch.clamp(image.view(height, width, 3), 0, 1)

In [ ]:
# ============================================
# LOSS FUNCTIONS
# ============================================
def l1_loss(pred, target):
    return (pred - target).abs().mean()

def ssim_loss(pred, target):
    C1, C2 = 0.01**2, 0.03**2
    mu_x, mu_y = pred.mean(), target.mean()
    sigma_x = ((pred - mu_x)**2).mean()
    sigma_y = ((target - mu_y)**2).mean()
    sigma_xy = ((pred - mu_x) * (target - mu_y)).mean()
    ssim = ((2*mu_x*mu_y + C1) * (2*sigma_xy + C2)) / ((mu_x**2 + mu_y**2 + C1) * (sigma_x + sigma_y + C2))
    return 1 - ssim

def combined_loss(pred, target, lambda_ssim=0.2):
    return (1-lambda_ssim)*l1_loss(pred, target) + lambda_ssim*ssim_loss(pred, target)

def psnr(pred, target):
    return 10 * torch.log10(1.0 / ((pred - target) ** 2).mean())

In [ ]:
# ============================================
# DENSIFICATION & PRUNING (Notebook 06)
# ============================================
def densify_and_prune(model, config):
    """Adaptive density control."""
    grad_avg = model.xyz_grad_accum / (model.denom + 1e-6)
    needs_densify = grad_avg > config['densify_grad_thresh']
    is_small = model.scales.max(dim=-1).values < 0.01
    
    # Clone small Gaussians
    to_clone = needs_densify & is_small
    if to_clone.sum() > 0:
        idx = to_clone.nonzero(as_tuple=True)[0]
        model._xyz = nn.Parameter(torch.cat([model._xyz.data, model._xyz.data[idx] + torch.randn_like(model._xyz.data[idx]) * 0.01]))
        model._log_scale = nn.Parameter(torch.cat([model._log_scale.data, model._log_scale.data[idx]]))
        model._rotation = nn.Parameter(torch.cat([model._rotation.data, model._rotation.data[idx]]))
        model._opacity = nn.Parameter(torch.cat([model._opacity.data, model._opacity.data[idx]]))
        model._sh = nn.Parameter(torch.cat([model._sh.data, model._sh.data[idx]]))
    
    # Split large Gaussians
    to_split = needs_densify & ~is_small
    if to_split.sum() > 0:
        idx = to_split.nonzero(as_tuple=True)[0]
        n = len(idx)
        offset = model.scales[idx] * 0.5
        new_xyz1 = model._xyz.data[idx] + torch.randn(n, 3, device=device) * offset
        new_xyz2 = model._xyz.data[idx] - torch.randn(n, 3, device=device) * offset
        new_scale = model._log_scale.data[idx] - 0.5
        model._xyz = nn.Parameter(torch.cat([model._xyz.data, new_xyz1, new_xyz2]))
        model._log_scale = nn.Parameter(torch.cat([model._log_scale.data, new_scale, new_scale]))
        model._rotation = nn.Parameter(torch.cat([model._rotation.data, model._rotation.data[idx], model._rotation.data[idx]]))
        model._opacity = nn.Parameter(torch.cat([model._opacity.data, model._opacity.data[idx], model._opacity.data[idx]]))
        model._sh = nn.Parameter(torch.cat([model._sh.data, model._sh.data[idx], model._sh.data[idx]]))
    
    # Prune low-opacity
    keep = model.opacity.squeeze() > config['prune_opacity_thresh']
    if not keep.all():
        model._xyz = nn.Parameter(model._xyz.data[keep])
        model._log_scale = nn.Parameter(model._log_scale.data[keep])
        model._rotation = nn.Parameter(model._rotation.data[keep])
        model._opacity = nn.Parameter(model._opacity.data[keep])
        model._sh = nn.Parameter(model._sh.data[keep])
    
    # Reset accumulators
    n = len(model)
    model.xyz_grad_accum = torch.zeros(n, device=device)
    model.denom = torch.zeros(n, device=device)
    return n

In [ ]:
# ============================================
# CHECKPOINTING
# ============================================
def save_checkpoint(model, iteration, loss, path):
    torch.save({
        'iteration': iteration, 'loss': loss, 'num_gaussians': len(model),
        'xyz': model._xyz.data, 'log_scale': model._log_scale.data,
        'rotation': model._rotation.data, 'opacity': model._opacity.data,
        'sh': model._sh.data
    }, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    model = GaussianModel(ckpt['num_gaussians'], sh_degree=config['sh_degree'], device=device)
    model._xyz = nn.Parameter(ckpt['xyz'].to(device))
    model._log_scale = nn.Parameter(ckpt['log_scale'].to(device))
    model._rotation = nn.Parameter(ckpt['rotation'].to(device))
    model._opacity = nn.Parameter(ckpt['opacity'].to(device))
    model._sh = nn.Parameter(ckpt['sh'].to(device))
    model.xyz_grad_accum = torch.zeros(len(model), device=device)
    model.denom = torch.zeros(len(model), device=device)
    print(f"📂 Loaded {ckpt['num_gaussians']} Gaussians @ iter {ckpt['iteration']}")
    return model, ckpt['iteration']

In [ ]:
# ============================================
# TRAINING
# ============================================
def train(model, cameras, images, config, start_iter=0):
    optimizer = Adam([
        {'params': [model._xyz], 'lr': config['lr_position']},
        {'params': [model._log_scale], 'lr': config['lr_scale']},
        {'params': [model._rotation], 'lr': config['lr_rotation']},
        {'params': [model._opacity], 'lr': config['lr_opacity']},
        {'params': [model._sh], 'lr': config['lr_sh']},
    ])
    
    history = {'loss': [], 'psnr': [], 'gaussians': []}
    
    for it in tqdm(range(start_iter, config['num_iterations']), desc='Training'):
        idx = np.random.randint(len(cameras))
        target = images[idx]
        cam = cameras[idx]
        
        optimizer.zero_grad()
        rendered = render_gaussians(model, cam)
        loss = combined_loss(rendered, target, config['lambda_ssim'])
        loss.backward()
        
        # Accumulate gradients for densification
        if model._xyz.grad is not None:
            grad_norm = model._xyz.grad.norm(dim=-1)
            n = min(len(grad_norm), len(model.xyz_grad_accum))
            model.xyz_grad_accum[:n] += grad_norm[:n]
            model.denom[:n] += 1
        
        optimizer.step()
        
        with torch.no_grad():
            p = psnr(rendered, target)
        history['loss'].append(loss.item())
        history['psnr'].append(p.item())
        history['gaussians'].append(len(model))
        
        # Densification
        if config['densify_from'] <= it < config['densify_until'] and it % config['densify_every'] == 0:
            densify_and_prune(model, config)
            optimizer = Adam([
                {'params': [model._xyz], 'lr': config['lr_position']},
                {'params': [model._log_scale], 'lr': config['lr_scale']},
                {'params': [model._rotation], 'lr': config['lr_rotation']},
                {'params': [model._opacity], 'lr': config['lr_opacity']},
                {'params': [model._sh], 'lr': config['lr_sh']},
            ])
        
        # Display
        if (it + 1) % config['display_every'] == 0:
            print(f"Iter {it+1}: Loss={loss.item():.4f}, PSNR={p.item():.2f}dB, Gaussians={len(model)}")
        
        # Checkpoint
        if (it + 1) % config['checkpoint_every'] == 0:
            save_checkpoint(model, it+1, loss.item(), f"{config['checkpoint_dir']}/ckpt_{it+1}.pth")
            with torch.no_grad():
                rendered = render_gaussians(model, cameras[0])
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            axes[0].imshow(rendered.cpu().numpy())
            axes[0].set_title(f'Rendered (Iter {it+1})')
            axes[1].imshow(images[0].cpu().numpy())
            axes[1].set_title('Ground Truth')
            for ax in axes: ax.axis('off')
            plt.savefig(f"{config['output_dir']}/compare_{it+1}.png")
            plt.show()
    
    return history

In [ ]:
# ============================================
# 🚀 RUN TRAINING
# ============================================
print("="*60)
print(f"🚀 3D GAUSSIAN SPLATTING - Training on {SCENE}")
print("="*60)

model = GaussianModel(config['initial_gaussians'], sh_degree=config['sh_degree'], device=device)
print(f"✅ Model: {len(model)} Gaussians")

history = train(model, cameras, images, config)

# Save final
save_checkpoint(model, config['num_iterations'], history['loss'][-1], 
                f"{config['checkpoint_dir']}/final.pth")
print(f"\n✅ Training complete! Final: {len(model)} Gaussians, PSNR: {history['psnr'][-1]:.2f}dB")

In [ ]:
# ============================================
# 📊 VISUALIZATIONS
# ============================================
print("📊 Creating visualizations...")

# Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['loss'])
axes[0].set_yscale('log')
axes[0].set_title('Loss')
axes[1].plot(history['psnr'])
axes[1].set_title('PSNR (dB)')
axes[2].plot(history['gaussians'])
axes[2].set_title('Gaussians')
for ax in axes: ax.set_xlabel('Iteration')
plt.tight_layout()
plt.savefig(f"{config['output_dir']}/training_curves.png", dpi=150)
plt.show()

# Multi-view comparison
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    with torch.no_grad():
        rendered = render_gaussians(model, cameras[i])
    axes[0, i].imshow(rendered.cpu().numpy())
    axes[0, i].set_title(f'Rendered {i}')
    axes[1, i].imshow(images[i].cpu().numpy())
    axes[1, i].set_title(f'GT {i}')
for ax in axes.flat: ax.axis('off')
plt.suptitle(f'{SCENE} - Final Results')
plt.tight_layout()
plt.savefig(f"{config['output_dir']}/comparison.png", dpi=150)
plt.show()

# 3D visualization
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
n_show = min(3000, len(model))
with torch.no_grad():
    xyz = model.xyz[:n_show].cpu().numpy()
    cols = torch.clamp(model._sh[:n_show, 0, :] + 0.5, 0, 1).cpu().numpy()
ax.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=cols, s=1)
ax.set_title(f'3D Gaussians ({len(model)} total)')
plt.savefig(f"{config['output_dir']}/3d_view.png", dpi=150)
plt.show()

In [ ]:
# ============================================
# 🎬 CREATE ORBIT VIDEO
# ============================================
print("🎬 Creating orbit video...")

frames = []
for i in tqdm(range(60)):
    angle = 2 * np.pi * i / 60
    pos = np.array([4*np.cos(angle), 0.5, 4*np.sin(angle)])
    fwd = -pos / np.linalg.norm(pos)
    right = np.cross(fwd, [0,1,0])
    right = right / (np.linalg.norm(right) + 1e-6)
    up = np.cross(right, fwd)
    R = np.stack([right, -up, fwd], axis=0)
    
    size = 256
    focal = size / (2 * np.tan(np.radians(50) / 2))
    cam = Camera(torch.tensor(R, dtype=torch.float32), 
                 torch.tensor(pos, dtype=torch.float32),
                 focal, focal, size/2, size/2, size, size)
    
    with torch.no_grad():
        img = render_gaussians(model, cam)
    frames.append((img.cpu().numpy() * 255).astype(np.uint8))

video_path = f"{config['output_dir']}/orbit.mp4"
imageio.mimwrite(video_path, frames, fps=30)
print(f"✅ Video saved: {video_path}")

# Preview
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(frames[i * 6])
    ax.axis('off')
plt.suptitle('Orbit Video Frames')
plt.show()

In [ ]:
# ============================================
# 📋 SUMMARY
# ============================================
print("\n" + "="*60)
print("✅ ALL COMPLETE!")
print("="*60)
print(f"\n🎯 Scene: {SCENE}")
print(f"📊 Final Gaussians: {len(model)}")
print(f"📈 Final PSNR: {history['psnr'][-1]:.2f} dB")
print(f"📁 Results saved to: {config['output_dir']}")
print(f"\nFiles:")
print(f"  • final.pth")
print(f"  • training_curves.png")
print(f"  • comparison.png")
print(f"  • 3d_view.png")
print(f"  • orbit.mp4")
print("="*60)

In [ ]:
# ============================================
# 🔄 RESUME TRAINING (Optional)
# ============================================
# Uncomment to resume from checkpoint:

# model, start_iter = load_checkpoint(f"{config['checkpoint_dir']}/ckpt_10000.pth")
# config['num_iterations'] = 20000
# history = train(model, cameras, images, config, start_iter=start_iter)